In [1]:
import os
import sys
sys.path.append(os.path.abspath('../../../'))  # Adjust path to project root
from src.app.vectorstores.search_factory import SearchServiceFactory
from src.app.config import INDEX_NAME
from src.app.vectorstores.sample_data import get_sample_documents, get_documents_by_category, get_documents_by_tag

In [2]:
search_service = SearchServiceFactory.create_opensearch()
info = search_service.client.info()

print("Connection successful!")
print(f"Cluster name: {info['cluster_name']}")
print(f"OpenSearch version: {info['version']['number']}")
print("✓ OpenSearch Service created successfully")


Connection successful!
Cluster name: docker-cluster
OpenSearch version: 3.4.0
✓ OpenSearch Service created successfully


In [3]:
test_search_query = "I wanna do some migrations regarding argo cd, i want it's build and pubishh"

# Full-text search (BM25)

In [4]:

print("\nTesting Full-Text Search (BM25)...")
print("\nSearch Query:", test_search_query)

results = search_service.full_text_search(
    index_name=INDEX_NAME,
    query=test_search_query,
    fields=["title", "content", "category"],
    size=3
)

for i, result in enumerate(results, 1):
    print(f"\n  {i}. {result['source']['title']}")
    print(f"     Score: {result['score']:.4f}")
    print(f"     Category: {result['source']['category']}")




Testing Full-Text Search (BM25)...

Search Query: I wanna do some migrations regarding argo cd, i want it's build and pubishh

  1. Argo CD Build Migration Template
     Score: 2.4730
     Category: migration-templates

  2. Argo CD Deployment Migration Template
     Score: 2.2750
     Category: migration-templates

  3. Docker Build and Push Migration Template
     Score: 0.9474
     Category: migration-templates


# Vector search

In [5]:
print("\nTesting Vector Search (Semantic)...")
print("\nSearch Query:", test_search_query)

query_vector = search_service.embed(test_search_query)

results = search_service.vector_search(
    index_name=INDEX_NAME,
    vector=query_vector,
    vector_field="embedding",
    size=3
)
print(f"Found {len(results)} results:")
for i, result in enumerate(results, 1):
    print(f"\n  {i}. {result['source']['title']}")
    print(f"     Score: {result['score']:.4f}")
    print(f"     Tags: {', '.join(result['source']['tags'])}")




Testing Vector Search (Semantic)...

Search Query: I wanna do some migrations regarding argo cd, i want it's build and pubishh
Found 3 results:

  1. Argo CD Build Migration Template
     Score: 0.7990
     Tags: argocd, docker, build, containers, migration

  2. Argo CD Deployment Migration Template
     Score: 0.7942
     Tags: argocd, deployment, kubernetes, gitops, migration

  3. Docker Build and Push Migration Template
     Score: 0.6603
     Tags: docker, containers, build, registry, migration


# Hybrid search (BM25 + Vector)

In [6]:
print("\nTesting Full-Text Search (BM25) + Vector Search (Semantic)...")
print("\nSearch Query:", test_search_query)

query_vector = search_service.embed(test_search_query)

results = search_service.hybrid_search(
    index_name=INDEX_NAME,
    query=test_search_query,
    vector=query_vector,
    text_fields=["title", "content", "category", "tags"],
    vector_field="embedding",
    text_weight=0.5,
    vector_weight=0.5,
    size=5
)
print(f"Found {len(results)} results:")
for i, result in enumerate(results, 1):
    print(f"\n  {i}. {result['source']['title']}")
    print(f"     Score: {result['score']:.4f}")
    print(f"     Category: {result['source']['category']}")
    print(f"     Author: {result['source']['author']}")


Testing Full-Text Search (BM25) + Vector Search (Semantic)...

Search Query: I wanna do some migrations regarding argo cd, i want it's build and pubishh
Found 5 results:

  1. Argo CD Build Migration Template
     Score: 1.6360
     Category: migration-templates
     Author: Migration Team

  2. Argo CD Deployment Migration Template
     Score: 1.5346
     Category: migration-templates
     Author: Migration Team

  3. Docker Build and Push Migration Template
     Score: 0.8038
     Category: migration-templates
     Author: Migration Team

  4. Gradle Build Migration Template
     Score: 0.5611
     Category: migration-templates
     Author: Migration Team

  5. Python Build Migration Template
     Score: 0.5417
     Category: migration-templates
     Author: Migration Team
